# POSEIDON workflow example

<p align="center"><img src="../docs/logo.png" width="300"></p>

This notebook walks through the POSEIDON processing chain on one Carolina
Beach flood event, using the same library calls as the HPC drivers in
`poseidon_deploy/naiads`:

1. **Grid** - interpolate ground LiDAR onto the site's rectification grid.
2. **Rectify** - project camera images and segmentation labels onto the grid.
3. **Depths** - estimate water surface elevation (WSE) and depth per pond.
4. **Roadway** - summarize flooding inside a roadway region of interest.

Outputs go to `data/notebook_example/` (git-ignored). Site values (camera
calibration, grid extent, CRS) come from
`poseidon_deploy/configs/sites/carolina_beach.yaml`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import poseidon_core
import poseidon_utils

poseidon_utils.setup_logging()  # show progress messages in the notebook

In [ ]:
REPO = Path.cwd().parent
site = poseidon_utils.load_site_config("down_east")
EVENT = REPO / "data/flood_events/CB_03_20231002131313_20231002203713"
OUT = REPO / "data/notebook_example"
N_IMAGES = 4  # images to process in this demo

ROI_JSON = site.resolve(site.roi_json)
if not ROI_JSON.exists():  # local copies may sit directly in data/
    ROI_JSON = REPO / "data" / ROI_JSON.name
print(site.name, site.camera_name, site.grid)

## 1. Elevation grid from LiDAR

Ground-classified LiDAR points inside the site extent are linearly
interpolated onto a regular grid (rows run south to north). On the HPC this
is stage 06 (`run_prep_grid.py`), run once per site.

In [ ]:
grid = site.grid
generator = poseidon_core.GridGenerator(
    site.resolve(site.lidar_file), grid.min_x, grid.max_x, grid.min_y,
    grid.max_y, extent_units="meters", lidar_units=site.lidar_units)
grid_x, grid_y, grid_z = generator.gen_grid(
    grid.resolution, generator.create_point_array(), dir=str(OUT / "grids"),
    grid_descriptor=site.name)
generator.plot_elev_grid(grid_z);

## 2. Rectify images and labels

Each grid cell is projected into the camera once; rectifying an image then
just samples the pixel under every cell. `use_gpu=None` uses CuPy when a
GPU is available and NumPy otherwise.

In [ ]:
rectifier = poseidon_core.ImageRectifier(
    site.intrinsics, site.extrinsics, grid_x, grid_y, grid_z, use_gpu=None)

image_path = sorted((EVENT / "orig_images").iterdir())[0]
label_path = sorted((EVENT / "labels").iterdir())[0]
to_host = poseidon_core.backend.to_numpy
rectified = to_host(rectifier.merge_rectify(image_path))
rectified_labels = to_host(rectifier.merge_rectify(label_path, labels=True))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(rectified, origin="lower")
axes[0].set_title("Rectified image")
axes[1].imshow(rectified_labels[..., 0] == 1, origin="lower", cmap="Blues")
axes[1].set_title("Rectified water mask")
for ax in axes:
    ax.set_axis_off()

## 3. Water surface elevation and depth

Connected water regions ("ponds") are found in the rectified mask. Each
pond's WSE is a statistic of the ground elevation along its waterline (its
inner boundary); depth is WSE minus ground inside the pond.

In [ ]:
processor = poseidon_core.DepthMapProcessor(grid_z, plot_edges=False,
                                            use_gpu=None)
maps = processor.process_single_depth_map(rectified_labels, "demo")
depth = to_host(maps["demo_depth_map_95_perc"])
wse = to_host(maps["demo_wse_map_95_perc"])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, data, title in ((axes[0], wse, "WSE (m)"),
                        (axes[1], depth, "Depth (m)")):
    image = ax.imshow(data, origin="lower")
    fig.colorbar(image, ax=ax, shrink=0.7, label=title)
    ax.set_axis_off()
print(f"Maximum depth: {np.nanmax(depth):.3f} m")

## 4. Process a small event and summarize the roadway

The batch methods write single-file ZIP Zarr stores, as the HPC stages
07-10 do. Re-running replaces earlier outputs.

In [ ]:
import shutil

event = OUT / EVENT.name
for sub in ("orig_images", "labels"):
    (event / sub).mkdir(parents=True, exist_ok=True)
    for path in sorted((EVENT / sub).iterdir())[:N_IMAGES]:
        shutil.copy(path, event / sub)

zarr_dir = event / "zarr"
rectifier.merge_rectify_folder(event / "labels", zarr_dir / "labels_rects",
                               labels=True)
processor.process_depth_maps(zarr_dir / "labels_rects.zip",
                             zarr_dir / "depth_maps.zip")

analyzer = poseidon_core.RoadwayAnalyzer(event, ROI_JSON)
analyzer.process_single_event()

In [ ]:
import pandas as pd

csv = event / f"{analyzer.roi_name}_accessibility_time_series.csv"
access = pd.read_csv(csv)
access["Coverage (%)"] = 100 * access["FloodedPixels"] / access["TotalPixels"]
access[["Time", "Impassable", "Coverage (%)", "MaxDepth", "MeanDepth"]]

## Next steps

* **Flood events** - `poseidon_utils.EventTracker` downloads Sunny Day
  Flooding Project water levels and builds the event CSVs (needs API
  credentials).
* **Images** - `poseidon_utils.ImageHandler` pulls event images from the
  SDFP archive and filters them to daylight.
* **HPC** - `poseidon_deploy/hpc_submission_scripts` runs every stage as
  LSF jobs; see `docs/README.md`.